# Lesson 3.2 - Background Jobs and Runbooks

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Run several PowerShell tasks in the background
- Collect and combine job output
- Sketch a runbook-style script for later automation

**Project file(s) produced:** `scripts/lesson-3.2-jobs-runbooks.ps1` and `runbooks/Invoke-InventoryCleanup.ps1`

## Concepts

A **background job** lets PowerShell do work without blocking the current prompt. That is useful when you need parallel checks or long-running tasks.

A **runbook** is an automation script designed to run in a managed environment such as Azure Automation. Even before you deploy one, it helps to write scripts in a runbook-friendly style: explicit inputs, clear output, and predictable side effects.

Reference: [Microsoft Learn: Automate Administration with PowerShell](https://learn.microsoft.com/en-us/training/courses/az-040t00)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import textwrap

lesson_dir = Path('study-plan/topic-powershell/phase-3-modules-automation')
runbook_dir = lesson_dir / 'runbooks'
artifact_dir = lesson_dir / 'artifacts'
runbook_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

def run_powershell(script_text: str, script_name: str) -> int:
    script_path = lesson_dir / script_name
    script_path.write_text(script_text, encoding='utf-8')
    executable = shutil.which('pwsh') or shutil.which('powershell')
    if executable:
        result = subprocess.run([executable, '-NoLogo', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(script_path)], capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
        return result.returncode
    print(f'{script_path.name} written. Run it in PowerShell to see the live output.')
    return 0

## Code-Along: Step 1 - Fan out work with jobs

Here the script launches a few background jobs and then gathers their output. This is a simple pattern for running repeated checks in parallel.

In [ ]:
script = textwrap.dedent(r'''
$names = 'alpha', 'beta', 'gamma'
$jobs = foreach ($name in $names) {
    Start-Job -ScriptBlock { param($item) "Processed: $item" } -ArgumentList $name
}
Wait-Job -Job $jobs | Out-Null
$results = Receive-Job -Job $jobs
$results | Set-Content -Path (Join-Path $PSScriptRoot 'artifacts/job-results.txt')
$results
Remove-Job -Job $jobs
''')
run_powershell(script, 'lesson-3.2-background-jobs.ps1')

## Code-Along: Step 2 - Draft a runbook

A runbook should be readable on its own. This step creates a script skeleton that makes it obvious where inputs, logging, and operational commands would live.

In [ ]:
runbook_path = runbook_dir / 'Invoke-InventoryCleanup.ps1'
runbook_content = textwrap.dedent(r'''
param(
    [Parameter(Mandatory)]
    [string]$WorkspacePath
)

# TODO: add validation, logging, and cleanup operations for an Azure Automation runbook
Write-Host "Preparing cleanup for $WorkspacePath"
''')
runbook_path.write_text(runbook_content, encoding='utf-8')
print(runbook_path.read_text(encoding='utf-8'))

## Exercise

Change the job fan-out so it handles five items instead of three and writes a short timestamp next to each result.

**Your task:**
1. Update the item list.
2. Include a timestamp in the job output.
3. Confirm the text file still captures every result.

**Expected result:** a background-job workflow that can scale to more items without changing its structure.

In [ ]:
# TODO: expand the job list and include a timestamp in each result
exercise_script = textwrap.dedent(r'''
Write-Host "TODO: expand the background job example"
''')
run_powershell(exercise_script, 'lesson-3.2-exercise.ps1')

## Key Takeaways

- Background jobs are useful when a task can be split into independent pieces.
- A runbook is just a disciplined script that is ready for managed automation.
- Saving job output makes it easier to combine results later.

---

**Next:** [Lesson 3.3 - Testing and Quality](./lesson-3.3-testing-quality.ipynb)